<a href="https://colab.research.google.com/github/viktorbrojs-sys/BH_DS_Pro/blob/main/HW_1/ai4i2020_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Предсказание отказов оборудования

**Датасет:** AI4I 2020 Predictive Maintenance Dataset  
**Задача:** Бинарная классификация (Machine failure: 0/1)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
import joblib
from IPython.display import display

sns.set_style('whitegrid')
%matplotlib inline

## 1. Загрузка данных

In [ ]:
df = pd.read_csv('ai4i2020.csv')
print("Датасет загружен. Размер:", df.shape)

## 2. Первичный осмотр данных

In [ ]:
print("\nИнформация о данных:")
df.info()
print("\nПервые 5 строк:")
display(df.head())
print("\nПроверка пропусков:")
print(df.isnull().sum())

## 3. Анализ целевой переменной

In [ ]:
print("\nРаспределение отказов (Machine failure):")
failure_counts = df['Machine failure'].value_counts()
print(failure_counts)
print(f"Доля отказов: {failure_counts[1] / len(df) * 100:.2f}%")

## 4. Визуализация распределений признаков

In [ ]:
numeric_features = ['Air temperature [K]', 'Process temperature [K]',
                    'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']

for feature in numeric_features:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x=feature, hue='Machine failure', bins=50, alpha=0.5, kde=True)
    plt.title(f'Распределение {feature} по классам отказа')
    plt.show()

In [ ]:
for feature in numeric_features:
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=df, x='Machine failure', y=feature)
    plt.title(f'Сравнение {feature} для классов отказа')
    plt.show()

## 5. Корреляционная матрица

In [ ]:
corr_features = numeric_features + ['Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
corr_matrix = df[corr_features].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Корреляционная матрица')
plt.show()

## 6. Feature Engineering

In [ ]:
df['Temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['Power'] = df['Rotational speed [rpm]'] * df['Torque [Nm]']
df['Stress'] = df['Tool wear [min]'] * df['Torque [Nm]']
print("Добавлены новые признаки. Теперь колонок:", df.shape[1])

## 7. Предобработка данных

In [ ]:
# Удаление лишних колонок
cols_to_drop = ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
df.drop(columns=cols_to_drop, inplace=True)
print("Оставшиеся колонки:", df.columns.tolist())

# Кодирование категориального признака Type
type_mapping = {'L': 0, 'M': 1, 'H': 2}
df['Type_encoded'] = df['Type'].map(type_mapping)
df.drop(columns=['Type'], inplace=True)
print("Кодирование Type выполнено.")

## 8. Подготовка данных для моделирования

In [ ]:
# Разделение на признаки и целевую переменную
X = df.drop('Machine failure', axis=1)
y = df['Machine failure']
print("Размер X:", X.shape)
print("Размер y:", y.shape)

# Разделение на train/test (стратифицированное)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Обучающая выборка: {X_train.shape}, отказов: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Тестовая выборка: {X_test.shape}, отказов: {y_test.sum()} ({y_test.mean()*100:.2f}%)")

In [ ]:
# Масштабирование числовых признаков
num_cols = ['Air temperature [K]', 'Process temperature [K]',
            'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]',
            'Temp_diff', 'Power', 'Stress']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])
print("Масштабирование выполнено.")

## 9. Обучение моделей

In [ ]:
# Логистическая регрессия
lr_params = {
    'C': [0.01, 0.1, 1, 10],
    'class_weight': ['balanced', None],
    'max_iter': [1000]
}
lr = LogisticRegression(random_state=42)
lr_grid = GridSearchCV(lr, lr_params, cv=StratifiedKFold(5), scoring='f1', n_jobs=-1, verbose=1)
print("Обучение логистической регрессии...")
lr_grid.fit(X_train_scaled, y_train)
print("Лучшие параметры (LR):", lr_grid.best_params_)
print("Лучший F1 на CV: {:.4f}".format(lr_grid.best_score_))

In [ ]:
# Случайный лес
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [5, 10, None],
    'class_weight': ['balanced', None]
}
rf = RandomForestClassifier(random_state=42)
rf_grid = GridSearchCV(rf, rf_params, cv=StratifiedKFold(5), scoring='f1', n_jobs=-1, verbose=1)
print("\nОбучение случайного леса...")
rf_grid.fit(X_train_scaled, y_train)
print("Лучшие параметры (RF):", rf_grid.best_params_)
print("Лучший F1 на CV: {:.4f}".format(rf_grid.best_score_))

## 10. Оценка на тестовой выборке

In [ ]:
models = {
    'Logistic Regression': lr_grid.best_estimator_,
    'Random Forest': rf_grid.best_estimator_
}

results = []
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba)

    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'ROC-AUC': roc_auc
    })

    cm = confusion_matrix(y_test, y_pred)
    print(f"\n{name}")
    print(f"Матрица ошибок:\n{cm}")
    print(classification_report(y_test, y_pred, digits=4))

results_df = pd.DataFrame(results)
print("\nСводная таблица результатов:")
display(results_df)

## 11. ROC-кривые

In [ ]:
plt.figure(figsize=(8, 6))
for name, model in models.items():
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Случайная модель')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-кривые')
plt.legend()
plt.grid()
plt.show()

## 12. Важность признаков (Random Forest)

In [ ]:
feature_names = X_train.columns
importances = rf_grid.best_estimator_.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.title("Важность признаков (Random Forest)")
plt.bar(range(len(importances)), importances[indices], align='center')
plt.xticks(range(len(importances)), feature_names[indices], rotation=45)
plt.tight_layout()
plt.show()

print("Рейтинг признаков:")
for i in indices:
    print(f"{feature_names[i]}: {importances[i]:.4f}")

## 13. Сохранение лучшей модели

In [ ]:
best_model = rf_grid.best_estimator_
joblib.dump(best_model, 'best_model_rf.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("Модель сохранена: best_model_rf.pkl")
print("Скейлер сохранён: scaler.pkl")